In [ ]:
from smolagents import CodeAgent
from smolagents.models import InferenceClientModel

model = InferenceClientModel(
    model_id="meta-llama/Llama-3.3-70B-Instruct", 
    token="**********"
)

agent = CodeAgent(tools=[], model=model, add_base_tools=True)

print(agent.run("Give me the 50th Fibonacci number"))

In [ ]:
from smolagents import CodeAgent, InferenceClientModel, DuckDuckGoSearchTool

# 1. Define constants once
HF_TOKEN = "*********"
MODEL_ID = "meta-llama/Llama-3.3-70B-Instruct"

# 2. Initialize model (model_id referenced only once)
model = InferenceClientModel(
    model_id=MODEL_ID,
    token=HF_TOKEN
)

# 3. Use the built-in DuckDuckGo web search tool
search_tool = DuckDuckGoSearchTool()

# 4. Create the agent with only the web search tool
agent = CodeAgent(
    tools=[search_tool],
    model=model,
    add_base_tools=False  # Only include the web search tool
)

# 5. Run a query
result = agent.run("What is the capital of Brazil?")
print(result)



In [ ]:
from smolagents import (
    CodeAgent,
    InferenceClientModel,
    DuckDuckGoSearchTool,
    VisitWebpageTool,
    PythonInterpreterTool
)

HF_TOKEN = "*********"
MODEL_ID = "meta-llama/Llama-3.3-70B-Instruct"

model = InferenceClientModel(
    model_id=MODEL_ID,
    token=HF_TOKEN
)

tools = [
    DuckDuckGoSearchTool(),
    VisitWebpageTool(),
    PythonInterpreterTool()
]

# Note: don't pass verbose — instead, set stream_outputs=False
agent = CodeAgent(
    tools=tools,
    model=model,
    add_base_tools=False,
    stream_outputs=False
)

while True:
    user_input = input("You: ")
    if user_input.lower() in {"quit", "exit"}:
        break

    # Run agent and only capture the final answer
    answer = agent.run(user_input)
    print("Agent:", answer)

In [ ]:
import os
from smolagents import CodeAgent, Tool
from smolagents.models import InferenceClientModel
from sentence_transformers import SentenceTransformer
import faiss
from pypdf import PdfReader

def load_pdf(path, chunk_size=500, overlap=50):
    reader = PdfReader(path)
    text = ""
    for page in reader.pages:
        page_text = page.extract_text()
        if page_text:
            text += page_text + "\n"
    chunks = []
    for i in range(0, len(text), chunk_size - overlap):
        chunks.append(text[i:i+chunk_size])
    return chunks

class RetrieverTool(Tool):
    name = "pdf_retriever"
    description = "Retrieve relevant context from the loaded PDF for answering questions."
    inputs = {
        "query": {
            "type": "string",
            "description": "The natural language question to retrieve context for."
        }
    }
    output_type = "string"

    def __init__(self, pdf_path, top_k: int = 3):
        super().__init__()
        self.top_k = top_k
        self.embedder = SentenceTransformer("all-MiniLM-L6-v2")
        self.chunks = load_pdf(pdf_path)
        embeddings = self.embedder.encode(self.chunks, convert_to_numpy=True)
        self.index = faiss.IndexFlatL2(embeddings.shape[1])
        self.index.add(embeddings)

    def forward(self, query: str) -> str:
        query_emb = self.embedder.encode([query], convert_to_numpy=True)
        scores, ids = self.index.search(query_emb, self.top_k)
        return "\n\n".join([self.chunks[i] for i in ids[0]])

model = InferenceClientModel(
    model_id="meta-llama/Llama-3.3-70B-Instruct",
    token=os.getenv("HF_TOKEN")
)

retriever = RetrieverTool("le.pdf", top_k=3)

agent = CodeAgent(
    tools=[retriever],
    model=model
)

print("📚 Chat with your PDF (type 'exit' to quit)\n")

while True:
    user_input = input("You: ")
    if user_input.lower() in ["exit", "quit"]:
        print("👋 Goodbye!")
        break
    try:
        response = agent.run(user_input)
        print("\nAgent:", response, "\n")
    except Exception as e:
        print(f"⚠️ Error: {e}")


In [ ]:
import os
from smolagents import CodeAgent, Tool
from smolagents.models import InferenceClientModel
from sentence_transformers import SentenceTransformer
import faiss
from pypdf import PdfReader

def load_pdf(path, chunk_size=500, overlap=50):
    reader = PdfReader(path)
    text = ""
    for page in reader.pages:
        page_text = page.extract_text()
        if page_text:
            text += page_text + "\n"
    chunks = []
    for i in range(0, len(text), chunk_size - overlap):
        chunks.append(text[i:i+chunk_size])
    return chunks

class RetrieverTool(Tool):
    name = "pdf_retriever"
    description = "Retrieve relevant context from the loaded PDF for answering questions."
    inputs = {
        "query": {
            "type": "string",
            "description": "The natural language question to retrieve context for."
        }
    }
    output_type = "string"

    def __init__(self, pdf_path, top_k: int = 3):
        super().__init__()
        self.top_k = top_k
        self.embedder = SentenceTransformer("all-MiniLM-L6-v2")
        self.chunks = load_pdf(pdf_path)
        embeddings = self.embedder.encode(self.chunks, convert_to_numpy=True)
        self.index = faiss.IndexFlatL2(embeddings.shape[1])
        self.index.add(embeddings)

    def forward(self, query: str) -> str:
        query_emb = self.embedder.encode([query], convert_to_numpy=True)
        scores, ids = self.index.search(query_emb, self.top_k)
        return "\n\n".join([self.chunks[i] for i in ids[0]])

model = InferenceClientModel(
    model_id="meta-llama/Llama-3.3-70B-Instruct",
    token=os.getenv("HF_TOKEN")
)

retriever = RetrieverTool("le.pdf", top_k=3)

agent = CodeAgent(
    tools=[retriever],
    model=model
)

print("📚 Chat with your PDF (type 'exit' to quit)\n")

while True:
    user_input = input("You: ")
    if user_input.lower() in ["exit", "quit"]:
        print("👋 Goodbye!")
        break
    try:
        response = agent.run(user_input)
        print("\nAgent:", response, "\n")
    except Exception as e:
        print(f"⚠️ Error: {e}")
